## Need to add the inital dependecies of the local ollama embedding model


In [2]:
import os

from dotenv import load_dotenv
from ollama import AsyncClient, Client
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances)

load_dotenv()

api_key = os.environ["OLLAMA_API_KEY"]
model = os.getenv("OLLAMA_MODEL", "kimi-k2.7-code:cloud")

client = AsyncClient(
    host="https://ollama.com",
    headers={"Authorization": f"Bearer {api_key}"}
)

# Local client for embeddings (no API key needed)
local_client = Client(host="http://localhost:11434")

EMBEDDING_MODEL = "nomic-embed-text:v1.5"

print(f"Chat model: {model}")
print(f"Embedding model: {EMBEDDING_MODEL}")



Chat model: kimi-k2.7-code:cloud
Embedding model: nomic-embed-text:v1.5


## Adding some dummy data to the document File.

In [3]:
documents = [
    {"id": 1,  "text": "Spring Boot simplifies building production-ready Java backend applications.",                                               "topic": "backend",           "level": "beginner"},
    {"id": 2,  "text": "Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.",                          "topic": "devops",            "level": "intermediate"},
    {"id": 3,  "text": "Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.",                                  "topic": "databases",         "level": "beginner"},
    {"id": 4,  "text": "Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.",                      "topic": "distributed-systems","level": "intermediate"},
    {"id": 5,  "text": "PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.",                  "topic": "databases",         "level": "beginner"},
    {"id": 6,  "text": "Vector databases store embeddings and make similarity search over high-dimensional vectors efficient.",                    "topic": "ai",                "level": "beginner"},
    {"id": 7,  "text": "RAG retrieves relevant external context and gives that context to a language model before generating an answer.",          "topic": "ai",                "level": "beginner"},
    {"id": 8,  "text": "An embedding model converts text into a dense numerical vector that captures semantic relationships.",                    "topic": "ai",                "level": "beginner"},
    {"id": 9,  "text": "Cosine similarity compares the angle between vectors and is commonly used for semantic text retrieval.",                  "topic": "ai",                "level": "intermediate"},
    {"id": 10, "text": "HNSW builds a navigable graph so approximate nearest-neighbor search can avoid scanning every vector.",                   "topic": "vector-search",     "level": "advanced"},
    {"id": 11, "text": "Exact nearest-neighbor search compares the query against every candidate vector to obtain the true top results.",         "topic": "vector-search",     "level": "intermediate"},
    {"id": 12, "text": "Quantization compresses vector representations so a vector database can reduce memory usage and improve search efficiency.","topic": "vector-search",     "level": "advanced"},
    {"id": 13, "text": "Payload indexes allow a vector database to combine semantic similarity with structured filters.",                         "topic": "vector-search",     "level": "advanced"},
    {"id": 14, "text": "Python is widely used for machine learning, data processing, and building AI applications.",                              "topic": "programming",       "level": "beginner"},
    {"id": 15, "text": "Async programming can improve throughput for applications that spend significant time waiting on network I/O.",           "topic": "programming",       "level": "intermediate"},
    {"id": 16, "text": "API gateways can provide routing, authentication, rate limiting, and observability for microservices.",                   "topic": "distributed-systems","level": "intermediate"},
    {"id": 17, "text": "Microservices split an application into independently deployable services that communicate over APIs or events.",         "topic": "distributed-systems","level": "intermediate"},
    {"id": 18, "text": "Observability combines logs, metrics, and traces to understand the behavior of distributed applications.",               "topic": "devops",            "level": "intermediate"},
    {"id": 19, "text": "A cache can reduce database load and latency when frequently requested data can be reused safely.",                       "topic": "databases",         "level": "beginner"},
    {"id": 20, "text": "Reranking can improve retrieval quality by applying a more expensive relevance model to an initial candidate set.",       "topic": "ai",                "level": "advanced"},
]

texts = [d["text"] for d in documents]
print(f"{len(documents)} documents")
print(texts[0])

20 documents
Spring Boot simplifies building production-ready Java backend applications.


In [4]:
# So we will do the text to vector embeddings

# def embed_documents(texts: list[str]) -> list[list[float]]:
#     resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
#     return [d.embedding for d in resp.data]

# Using the local sync client for embeddings (no async/await needed)
def embed_documents(texts: list[str]) -> list[list[float]]:
    resp = local_client.embed(model=EMBEDDING_MODEL, input=texts)
    return resp.embeddings

doc_embeddings = embed_documents(texts)

print("Number of embeddings:", len(doc_embeddings))
print("Dimensions per vector:", len(doc_embeddings[0]))
print("First 5 values:", doc_embeddings[0])


Number of embeddings: 20
Dimensions per vector: 768
First 5 values: [0.01587073, 0.0143490415, -0.15083241, 0.022391116, -0.0056411303, -0.044014655, 0.06838984, 0.020035489, 0.011719886, -0.06643646, -0.026697444, 0.020020306, 0.07856472, 0.051622365, -0.005417038, -0.052564975, -0.045639336, -0.046357777, 0.050830834, -0.051604025, 0.04275172, -0.043724716, -0.009307005, -0.009667708, 0.07892744, 0.067637675, -0.04469903, 0.031225227, -0.0049334113, -0.0050834753, 0.07322173, -0.011570471, -0.0014050967, -0.003627849, -0.029376937, -0.02712228, 0.008051553, -0.012051624, 0.02598523, -0.05746531, -0.02077018, -0.054761764, 0.034949616, -0.020360112, 0.07091764, 0.017492617, 0.08629436, 0.009887071, 0.032523595, -0.06577269, -0.018027311, -0.035755813, -0.01865462, -0.047877613, 0.03410743, 0.0072094607, 0.010341431, 0.020306835, -0.0029038992, -0.028596725, 0.06450929, 0.084889404, -0.038358364, 0.052993197, 0.03518917, 0.014868937, 0.021263374, 0.010199648, 0.032436132, -0.0008512375

In [5]:
query = "Which programing language is commonly used for AI applications?"

query_embedding = embed_documents([query])  # returns the list with one vector

print("Query : ", query)
print("Dimensions : ", len(query_embedding[0]))
print("Collection must use size = ", len(doc_embeddings[0]))


Query :  Which programing language is commonly used for AI applications?
Dimensions :  768
Collection must use size =  768


## When to use what

| Metric | Intuition | Better match |
|---|---|---|
| Cosine | Angle between vectors | Higher |
| Euclidean / L2 | Straight-line distance | Lower |
| Dot product | Alignment + magnitude | Higher |

In [6]:
cos = cosine_similarity(query_embedding, doc_embeddings)[0]
l2  = euclidean_distances(query_embedding, doc_embeddings)[0]

print(f"{'Cos':>6}  {'L2':>5}  Document")
print("-" * 70)
for doc, c, d in zip(documents, cos, l2):
    print(f"{c:6.3f}  {d:6.3f}  {doc}")

   Cos     L2  Document
----------------------------------------------------------------------
 0.569   0.929  {'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}
 0.457   1.042  {'id': 2, 'text': 'Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.', 'topic': 'devops', 'level': 'intermediate'}
 0.591   0.904  {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}
 0.530   0.969  {'id': 4, 'text': 'Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.', 'topic': 'distributed-systems', 'level': 'intermediate'}
 0.500   1.000  {'id': 5, 'text': 'PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.', 'topic': 'databases', 'level': 'beginner'}
 0.494   1.006  {'id': 6, '

## TO Get the Top K values. As in best search values.

In [7]:
def retrieve(query_emb, doc_embs, docs, top_k=2):
    scores = cosine_similarity(query_emb, doc_embs)[0]

    # pair each document with its score, sort highest first, take top_k
    scored = list(zip(docs, scores))
    # print(scored)
    scored.sort(key=lambda x: x[1], reverse=True)

    return [{"document": doc, "score": float(score)} for doc, score in scored[:top_k]]


print("Top-K results:\n")
for hit in retrieve(query_embedding, doc_embeddings, documents, top_k=6):
    print(f"{hit['score']:.3f}  {hit['document']}")

Top-K results:

0.782  {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}
0.591  {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}
0.574  {'id': 7, 'text': 'RAG retrieves relevant external context and gives that context to a language model before generating an answer.', 'topic': 'ai', 'level': 'beginner'}
0.569  {'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}
0.563  {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'}
0.558  {'id': 18, 'text': 'Observability combines logs, metrics, and traces to understand the behavior of distributed applications.', 'topic': 'devops'

---
## Quadrant vector database

In [11]:
from qdrant_client import QdrantClient, models

#qdrant = QdrantClient(":memory:")

qdrant = QdrantClient(url="http://localhost:6333")

In [12]:
COLLECTION = "rag_fundamentals"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),   # always read from the model, never hard-code
        distance=models.Distance.COSINE,
    ),
)

print("Created:", COLLECTION, "| size:", len(doc_embeddings[0]))

Created: rag_fundamentals | size: 768


## Notes 

1. Inserting and updating the points if needed. 
2. Array of points
3. Zip --> Basically if you want to return a tupple of both the object. 

In [13]:
points = [
    models.PointStruct(
        id=doc["id"],
        vector=vec,
        payload={"text": doc["text"], "topic": doc["topic"], "level": doc["level"]},
    )
    for doc, vec in zip(documents, doc_embeddings)
]

qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)
print(f"Inserted {len(points)} points.")

Inserted 20 points.


## Quadrant search

In [14]:
results = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_embedding[0],    # a single vector, not a list
    limit=5,
    with_payload=True,
).points

for p in results:
    print(p)

id=14 version=1 score=0.7824943 payload={'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=3 version=1 score=0.591115 payload={'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=7 version=1 score=0.57436526 payload={'text': 'RAG retrieves relevant external context and gives that context to a language model before generating an answer.', 'topic': 'ai', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=1 version=1 score=0.5687424 payload={'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=15 version=1 score=0.5629768 payload={'text': 'Async programming can improve through